In [1]:
import os, re, gc, warnings
warnings.filterwarnings('ignore')

# ── Config ────────────────────────────────────────────────────────────
VECTOR_STORE_DIR = '../vector_store/chroma'
COLLECTION_NAME  = 'complaints'
EMBED_MODEL      = 'sentence-transformers/all-MiniLM-L6-v2'
TOP_K            = 5

# ── Load retriever ────────────────────────────────────────────────────
from sentence_transformers import SentenceTransformer
import chromadb

print('Loading embedding model...')
model = SentenceTransformer(EMBED_MODEL)

print('Loading ChromaDB...')
client     = chromadb.PersistentClient(path=VECTOR_STORE_DIR)
collection = client.get_collection(COLLECTION_NAME)
print(f'Collection loaded: {collection.count():,} chunks')

# ── Retriever function ────────────────────────────────────────────────
def retrieve(query, k=TOP_K, product_filter=None):
    query_vec = model.encode([query], normalize_embeddings=True).tolist()
    where     = {'product_category': product_filter} if product_filter else None
    results   = collection.query(
        query_embeddings=query_vec,
        n_results=k,
        where=where,
        include=['documents', 'metadatas', 'distances']
    )
    chunks = []
    for doc, meta, dist in zip(
        results['documents'][0],
        results['metadatas'][0],
        results['distances'][0]
    ):
        chunks.append({
            'text':     doc,
            'metadata': meta,
            'score':    round(1 - dist, 4)
        })
    return chunks

# ── Prompt builder ────────────────────────────────────────────────────
PROMPT_TEMPLATE = """You are a financial analyst assistant for CrediTrust Financial.
Your task is to answer questions about customer complaints.
Use ONLY the following retrieved complaint excerpts to formulate your answer.
If the context does not contain enough information, clearly state that.
Be concise, factual, and cite patterns you observe across multiple complaints.

Context:
{context}

Question: {question}

Answer:"""

def build_prompt(question, chunks):
    parts = []
    for i, chunk in enumerate(chunks, 1):
        meta    = chunk['metadata']
        product = meta.get('product_category', 'Unknown')
        issue   = meta.get('issue', '')
        parts.append(f"[{i}] Product: {product} | Issue: {issue}\n{chunk['text']}")
    context = "\n\n---\n\n".join(parts)
    return PROMPT_TEMPLATE.format(context=context, question=question)

print('Retriever and prompt builder ready!')
print()

# ── Test retrieval ────────────────────────────────────────────────────
test_q = 'Why are customers unhappy with credit cards?'
chunks = retrieve(test_q)
print(f'Query: "{test_q}"')
print(f'Retrieved {len(chunks)} chunks:')
for c in chunks:
    print(f"  Score:{c['score']} | {c['metadata']['product_category']} | {c['metadata']['issue']}")

print()
print('Prompt preview (first 500 chars):')
print(build_prompt(test_q, chunks)[:500])

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading ChromaDB...
Collection loaded: 1,448 chunks
Retriever and prompt builder ready!

Query: "Why are customers unhappy with credit cards?"
Retrieved 5 chunks:
  Score:0.5892 | Credit Card | Billing disputes
  Score:0.5856 | Savings Account | Managing an account
  Score:0.5669 | Credit Card | Closing your account
  Score:0.5491 | Credit Card | Problem when making payments
  Score:0.5463 | Credit Card | Late fee

Prompt preview (first 500 chars):
You are a financial analyst assistant for CrediTrust Financial.
Your task is to answer questions about customer complaints.
Use ONLY the following retrieved complaint excerpts to formulate your answer.
If the context does not contain enough information, clearly state that.
Be concise, factual, and cite patterns you observe across multiple complaints.

Context:
[1] Product: Credit Card | Issue: Billing disputes
my statement arrived and upon review i had noticed charges that i did not make. futher


In [4]:
from groq import Groq

groq_client = Groq(api_key="YOUR_GROQ_KEY_HERE")

def generate(question, chunks):
    prompt = build_prompt(question, chunks)
    response = groq_client.chat.completions.create(
        model="llama3-8b-8192",   # free, fast, strong
        messages=[{"role": "user", "content": prompt}],
        max_tokens=1024,
        temperature=0.2,
    )
    return response.choices[0].message.content

def ask(question, product_filter=None):
    print(f'\n{"="*60}')
    print(f'Q: {question}')
    if product_filter:
        print(f'   Filter: {product_filter}')
    print('='*60)
    chunks = retrieve(question, product_filter=product_filter)
    answer = generate(question, chunks)
    print(f'\nANSWER:\n{answer}')
    print(f'\nSOURCES:')
    for i, c in enumerate(chunks[:3], 1):
        meta = c['metadata']
        print(f"  [{i}] Score:{c['score']} | {meta['product_category']} | {meta['issue']}")
    return {'question': question, 'answer': answer, 'sources': chunks}

print('Generator ready!')

Generator ready!


In [6]:
from groq import Groq

groq_client = Groq(api_key="YOUR_GROQ_KEY_HERE")

def generate(question, chunks):
    prompt = build_prompt(question, chunks)
    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=1024,
        temperature=0.2,
    )
    return response.choices[0].message.content

def ask(question, product_filter=None):
    print(f'\n{"="*60}')
    print(f'Q: {question}')
    if product_filter:
        print(f'   Filter: {product_filter}')
    print('='*60)
    chunks = retrieve(question, product_filter=product_filter)
    answer = generate(question, chunks)
    print(f'\nANSWER:\n{answer}')
    print(f'\nSOURCES:')
    for i, c in enumerate(chunks[:3], 1):
        meta = c['metadata']
        print(f"  [{i}] Score:{c['score']} | {meta['product_category']} | {meta['issue']}")
    return {'question': question, 'answer': answer, 'sources': chunks}

print('Generator ready! Running test...')

# Quick test
result = ask("Why are customers unhappy with credit cards?")

Generator ready! Running test...

Q: Why are customers unhappy with credit cards?

ANSWER:
Based on the retrieved complaint excerpts, customers are unhappy with credit cards at CrediTrust Financial due to the following reasons:

1. **Billing disputes**: Multiple customers have reported issues with unauthorized charges on their credit card statements (complaints 1 and 4). This suggests a pattern of billing errors or potential identity theft.
2. **Credit limit reduction**: Customers have experienced a reduction in their credit limit without a valid reason, such as missing payments (complaint 3).
3. **Difficulty in making payments**: Some customers have reported issues with making payments, including being lied to by customer service representatives (complaint 4).
4. **Late fees**: One customer was incorrectly charged a late fee and had their card canceled as a result (complaint 5).
5. **Poor customer service**: Customers have reported being given excuses by customer service representativ

In [7]:
questions = [
    ("Why are customers unhappy with credit cards?", None),
    ("What are the most common savings account complaints?", None),
    ("What fraud issues do credit card customers face?", "Credit Card"),
    ("Why do customers complain about money transfers?", None),
    ("What billing problems do personal loan customers report?", None),
    ("What should product teams prioritise to reduce credit card complaints?", "Credit Card"),
    ("What issues do customers have when closing their accounts?", None),
    ("Are there complaints about customer service responsiveness?", None),
]

results = []
for question, product_filter in questions:
    result = ask(question, product_filter)
    results.append(result)

print('\n✅ All 8 questions evaluated!')


Q: Why are customers unhappy with credit cards?

ANSWER:
Based on the provided complaint excerpts, customers are unhappy with credit cards for the following reasons:

1. **Billing disputes**: Multiple customers (complaints 1 and 4) have reported unauthorized charges on their credit cards, leading to disputes and frustration.
2. **Credit limit reduction**: One customer (complaint 3) experienced a reduction in credit limit without a valid reason, despite making timely payments.
3. **Poor customer service**: Several customers (complaints 1, 2, 4, and 5) have reported issues with customer service representatives, including lying, unhelpfulness, and unprofessional behavior.
4. **Bad business practices**: One customer (complaint 5) accused the credit card company of stealing from customers through late fees and bad business practices.

Overall, customers are unhappy with credit cards due to issues with billing, credit limits, and poor customer service, which can lead to financial stress and